# 03 · Regressions

Factor alphas with Newey-West t-stats, and Fama-MacBeth regressions with controls. `filinglens research backtest` runs the whole pre-registered grid and writes the report. This notebook is for exploring.

In [ ]:
import pandas as pd

from filinglens.config import get_settings
from filinglens.research.prices import add_characteristics, load_panel
from filinglens.research.signals import load_signals
from filinglens.research.synthetic import make_world

RESEARCH = get_settings().processed_dir / "research"
REAL = all(
    (RESEARCH / f).exists() for f in ("signals.parquet", "prices.parquet", "factors.parquet")
)
if REAL:
    signals = load_signals(RESEARCH / "signals.parquet")
    panel = add_characteristics(load_panel(RESEARCH / "prices.parquet"))
    factors = pd.read_parquet(RESEARCH / "factors.parquet")
    factors.index = pd.PeriodIndex(factors.pop("month"), freq="M")
else:
    print("No research data yet: using the SYNTHETIC world (pipeline check, not results).")
    world = make_world()
    signals, panel, factors = world.signals, add_characteristics(world.panel), world.factors
source = panel.attrs.get("source")
print(f"{signals['cik'].nunique()} firms, {len(signals)} signal rows, returns: {source}")

In [ ]:
from filinglens.research.factors import available_models
from filinglens.research.portfolios import PortfolioSpec, portfolio_returns
from filinglens.research.regressions import quantile_table, sharpe

port = portfolio_returns(signals, panel, PortfolioSpec(signal="sim_cosine_tf"))
models = available_models(factors)
table = quantile_table(port, factors, models)
(table.xs("alpha", level="stat") * 1e4).round(1)

In [ ]:
(table.xs("t", level="stat")).round(2)

In [ ]:
print(f"L/S Sharpe (annualized): {sharpe(port['LS']):.2f}")

Fama-MacBeth: each signal with controls, then classical and new signals together (the incremental-value test).

In [ ]:
from filinglens.research.backtest import BacktestConfig, orient, run_fama_macbeth

cfg = BacktestConfig(
    signals=[
        c for c in ["sim_cosine_tf", "sim_jaccard", "sim_embed", "n_new_risks"] if c in signals
    ]
)
fm = run_fama_macbeth(orient(signals, cfg.signals), panel, cfg)
fm.pivot(index="variable", columns="spec", values="t").round(2)